# Build SNP Nakeds. 
 - One Put Option per symbol

In [1]:
## THIS CELL SHOULD BE IN ALL VSCODE NOTEBOOKS ##

MARKET = "SNP"

# Set the root
from from_root import from_root # type: ignore
ROOT = from_root()

import pandas as pd # type: ignore
from loguru import logger # type: ignore

pd.options.display.max_columns = None
pd.set_option('display.precision', 2)

from pathlib import Path
import sys

# Add `src` and ROOT to _src.pth in .venv to allow imports in VS Code
from sysconfig import get_path

if "src" not in Path.cwd().parts:
    src_path = str(Path(get_path("purelib")) / "_src.pth")
    with open(src_path, "w") as f:
        f.write(str(ROOT / "src\n"))
        f.write(str(ROOT))
        if str(ROOT) not in sys.path:
            sys.path.insert(1, str(ROOT))

# Start the Jupyter loop
from ib_async import util # type: ignore

util.startLoop()

logger.add(sink=ROOT / "log" / "ztest.log", mode="w")

1

# Get marketprice and IV

In [2]:
import asyncio
import math

from ib_async import IB, Contract, Stock
from tqdm import tqdm

from ibfuncs import get_ib, qualify_me
from utils import chunk_me, clean_ib_util_df, split_symbol_price_iv, to_list

In [3]:
async def get_tick_data(ib: IB, c: Contract, delay: float = 6):
    """Gets tick-by-tick data

    Args:
        ib (IB): IB instance
        c (Contract): a contract
        delay (float, optional): delay to fill. Defaults to 6 secs.

    Returns:
        _type_: IB ticker
    """

    # Request tick-by-tick data for the given contract asynchronously
    ticker = await ib.reqTickersAsync(c)

    # Introduce an optional delay if specified
    await asyncio.sleep(delay)

    # Return the retrieved ticker data
    return ticker


async def get_market_data(ib: IB, c: Contract, sleep: float = 2):

    """Gets market price with implied volatility. Works also in closed market.

    Args:
        ib (IB): IB instance
        c (Contract): a contract
        sleep (float, optional): delay to fill. Defaults to 2 secs.

    Returns:
        _type_: IB tick_
    """

    tick = ib.reqMktData(c, genericTickList="106")
    try:
        await asyncio.sleep(sleep)
    finally:
        ib.cancelMktData(c)

    return tick


async def get_a_price_iv(ib, contract, sleep: float = 2) -> dict:
    """[async] Computes price and IV of a contract.

    OUTPUT: dict{localsymbol, price, iv}

    Could take up to 12 seconds in case live prices are not available"""

    mkt_data = await get_market_data(ib, contract, sleep)
    undPrice = mkt_data.marketPrice()

    if math.isnan(undPrice):
        undPrice = mkt_data.close
        if math.isnan(undPrice):
            tick_data = await get_tick_data(ib, contract)
            tick_data_price = tick_data[0].marketPrice()
            undPrice = (
                tick_data_price
                if not math.isnan(tick_data_price)
                else tick_data[0].close
            )
            if math.isnan(undPrice):
                logger.info(f"No price found for {contract.localSymbol}!")

    iv = mkt_data.impliedVolatility
    return {"localsymbol": contract.localSymbol, "price": undPrice, "iv": iv}


async def get_mkt_prices(
   ib:IB, contracts: list, chunk_size: int = 44, sleep: int = 7
) -> pd.DataFrame:
    """[async] A faster way to get market prices."""

    contracts = to_list(contracts)
    chunks = chunk_me(contracts, chunk_size)
    results = dict()

    for cts in tqdm(chunks, desc="Mkt prices with IVs"):
        tasks = [get_a_price_iv(ib, c, sleep) for c in cts]
        res = await asyncio.gather(*tasks)

        for r in res:
            symbol, price, iv = r.values()
            results[symbol] = (price, iv)

    df_prices = split_symbol_price_iv(results)
    df_prices = pd.merge(
        clean_ib_util_df(contracts).iloc[:, :6], df_prices, on="symbol"
    )

    # remove unnecessary columns (for secType == `STK`)
    keep_cols = ~(
        (df_prices == 0).all() | (df_prices == "").all() | df_prices.isnull().all()
    )

    df_prices = df_prices.loc[:, keep_cols[keep_cols is True].index]

    return df_prices


In [4]:
with get_ib(MARKET) as ib:
    d = {"TSLA": "NYSE", "MSFT": "NYSE", "AAPL": "NYSE"}
    contracts = {}  # store contracts here
    
    for ticker in list(d.keys()):
        contracts[ticker] = Stock(ticker, d[ticker], "USD")

    cts = list(contracts.values())

    ib.run(qualify_me(ib, cts))
    


Qualifying contracts: 100%|██████████| 3/3 [00:00<00:00,  5.30it/s]


In [5]:
with get_ib(MARKET) as ib:
    df = ib.run(get_mkt_prices(ib, cts))

Mkt prices with IVs: 100%|██████████| 1/1 [00:07<00:00,  7.02s/it]


KeyError: False

In [ ]:
df